In [ ]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Ejercicios")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

val sc = spark.sparkContext
println(s"✅ Entorno listo — Spark ${spark.version}")

Ejercicio 1 — Crear y explorar un RDD de temperaturas
Crea un RDD a partir de la siguiente lista de temperaturas en grados Celsius registradas durante una semana en distintas ciudades. Luego imprime cuántas particiones tiene el RDD y confirma de qué tipo es usando getClass.getSimpleName.

In [ ]:
val temperaturas = List(22.5, 18.0, 35.1, 12.3, 28.7, 9.4, 31.0, 25.5, 17.2, 33.8)
// Tu código aquí

val temperaturasRDD = sc.parallelize(temperaturas)

println(s"Tipo del RDD:       ${temperaturasRDD.getClass.getSimpleName}")
println(s"Número de partes:   ${temperaturasRDD.getNumPartitions}")
println(s"¿Se ha ejecutado?:  No — solo hemos definido el plan")


Ejercicio 2 — Filtrar temperaturas extremas

Usando el RDD del ejercicio anterior, aplica dos transformaciones encadenadas: primero filtra las temperaturas superiores a 20 grados, y sobre ese resultado filtra de nuevo las que sean inferiores a 30 grados. Recoge el resultado con collect() e imprímel

In [ ]:
val TemperaturaPrimerFiltro = temperaturasRDD.filter(temp => temp > 20.0)
println(s"¿Se ha ejecutado?:  No — seguimos definiendo el plan")
val TemperaturaSegundoFiltro = TemperaturaPrimerFiltro.filter(temp => temp < 30.0)
println(s"el resultado es: ${TemperaturaSegundoFiltro.collect().mkString(", ")}")

Ejercicio 3 — Transformación de unidades

A partir del mismo RDD de temperaturas, crea un nuevo RDD que convierta cada valor de Celsius a Fahrenheit usando la fórmula F = C * 9/5 + 32. Recoge el resultado e imprímelo redondeado a un decimal.

In [ ]:
import org.apache.log4j.{Level, Logger}

// Esto silencia los mensajes de INFO y WARN de Spark
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

In [ ]:
val TemperaturaF = temperaturasRDD.map(t => (t * 9/5) + 32)

println(s"Temperaturas en Fahrenheit: ${TemperaturaF.collect().map(t => f"$t%.1f").mkString(", ")}")

Ejercicio 4 — Acciones de estadística básica

Crea un RDD con los precios de los siguientes artículos de una tienda online y calcula, usando las acciones de Spark, el número total de artículos, el precio más alto y el precio más bajo. No uses collect() para esto: encuentra las acciones específicas que devuelven cada valor directamente.

In [ ]:
val precios = sc.parallelize(List(49.99, 12.50, 199.00, 7.99, 89.90, 34.99, 149.95, 22.00))

val totalPrecios = precios.count()
println(s"Total de precios: $totalPrecios")

val precioMasAlto = precios.reduce((a, b) => if (a > b) a else b)
println(f"Precio más alto: $$${precioMasAlto}%.2f")

val precioMasBajo = precios.reduce((a, b) => if (a < b) a else b)
println(f"Precio más bajo: $$${precioMasBajo}%.2f")

Ejercicio 5 — Entender la lazy evaluation

Este ejercicio es conceptual y de observación. Ejecuta las siguientes dos celdas por separado y observa en qué momento aparece actividad en el Spark UI (http://localhost:4040).

In [ ]:
val numeros = sc.parallelize(1 to 100)
val multiplosde7 = numeros.filter(n => n % 7 == 0)
val dobles = multiplosde7.map(n => n * 2)
println("Celda A ejecutada — ¿Hay un nuevo job en el Spark UI?")

In [ ]:
val resultado = dobles.collect()
println("Celda B ejecutada — ¿Y ahora en el Spark UI?")
println(s"Resultado: ${resultado.mkString(", ")}")

Ejercicio 6 — Pipeline de transformaciones sobre nombres

Crea un RDD a partir de la siguiente lista de nombres de empleados. Aplica en cadena estas tres transformaciones: convierte cada nombre a mayúsculas, filtra solo los que empiecen por la letra "A", y luego ordénalos alfabéticamente. Usa collect() y muestra el resultado.

In [ ]:
import $ivy.`org.apache.spark::spark-core:3.5.0`
import $ivy.`org.apache.spark::spark-sql:3.5.0`
import org.apache.spark.sql.SparkSession
import org.apache.log4j.{Level, Logger}

Logger.getLogger("org").setLevel(Level.ERROR)

val spark = SparkSession.builder()
  .appName("SolucionDefinitiva")
  .master("local[*]")
  // 1. FORZAMOS EL SERIALIZADOR ESTÁNDAR (Esto evita el error de Kryo)
  .config("spark.serializer", "org.apache.spark.serializer.JavaSerializer")
  // 2. PERMISOS DE JAVA (El parche necesario para Java 17+)
  .config("spark.driver.extraJavaOptions", 
    "--add-opens=java.base/java.lang=ALL-UNNAMED " +
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED")
  .config("spark.executor.extraJavaOptions", 
    "--add-opens=java.base/java.lang=ALL-UNNAMED " +
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED")
  .getOrCreate()

val sc = spark.sparkContext
println("✅ Spark configurado sin Kryo — Listo para procesar")

In [ ]:
// ... (Mantén tu configuración de SparkSession anterior)

val empleados = sc.parallelize(List(
  "ana garcía", "pedro López", "alba martínez", "carlos ruiz",
  "adriana vega", "beatriz soler", "antonio mora", "sara jiménez"
))

// 1. Spark hace el procesamiento pesado
val procesados = empleados
  .map(_.toUpperCase)
  .filter(n => n.startsWith("A"))

// 2. Traemos los datos a Scala (collect) y ordenamos "fuera" de Spark
val resultado = procesados.collect().sorted

println(s"Resultado ordenado sin errores: ${resultado.mkString(", ")}")

Ejercicio 7 — flatMap sobre líneas de texto

Dado el siguiente RDD de frases sobre tecnología, usa flatMap para obtener un RDD con todas las palabras individuales de todas las frases. Luego cuenta cuántas palabras hay en total usando una acción.

In [ ]:
val frases = sc.parallelize(List(
  "spark procesa datos a gran velocidad",
  "scala es el lenguaje nativo de spark",
  "los datos se procesan en memoria ram",
  "hadoop almacena datos en disco hdfs"
))
val palabras = frases.flatMap(_.split(" "))
val palabrasUnicas = palabras.collect().distinct

println(s"Total de palabras en toda la frase: ${palabrasUnicas.length}")

In [1]:
// 1. Cargamos librerías
import $ivy.`org.apache.spark::spark-core:3.5.0`
import $ivy.`org.apache.spark::spark-sql:3.5.0`
import org.apache.spark.sql.SparkSession

// 2. Cerramos cualquier rastro previo
SparkSession.getActiveSession.foreach(_.stop())

// 3. LA CONFIGURACIÓN DE "FUERZA BRUTA"
val spark = SparkSession.builder()
  .master("local[*]")
  .appName("SparkRealRDD")
  // Forzamos el serializador más básico que existe
  .config("spark.serializer", "org.apache.spark.serializer.JavaSerializer")
  // Desactivamos cualquier optimización que use Kryo
  .config("spark.kryo.registrationRequired", "false")
  .config("spark.shuffle.manager", "sort") 
  // Intentamos inyectar los permisos directamente al proceso de ejecución
  .config("spark.driver.extraJavaOptions", "-Djdk.attach.allowAttachSelf=true --add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
  .config("spark.executor.extraJavaOptions", "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED --add-opens=java.base/java.lang=ALL-UNNAMED --add-opens=java.base/java.util=ALL-UNNAMED")
  .getOrCreate()

val sc = spark.sparkContext

// 4. PRUEBA DE FUEGO (Código Spark RDD Puro)
val frases = sc.parallelize(List(
  "spark procesa datos a gran velocidad",
  "scala es el lenguaje nativo de spark",
  "los datos se procesan en memoria ram"
))

try {
    // Si esto funciona, hemos vencido al Shuffle
    val palabrasUnicasCount = frases
      .flatMap(_.split(" "))
      .distinct() // <--- El punto crítico
      .count()    // <--- La acción que dispara el Shuffle

    println("\n" + "!" * 40)
    println(s"¡LOGRADO! Spark RDD funcionó: $palabrasUnicasCount")
    println("!" * 40)
} catch {
    case e: Exception => 
      println("\n" + "X" * 40)
      println("Incluso con configuración extrema, Java 17 bloquea el Shuffle.")
      println("Error resumido: " + e.getMessage.take(100))
      println("X" * 40)
}

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/22 11:11:20 INFO SparkContext: Running Spark version 3.5.0
26/04/22 11:11:20 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/22 11:11:20 INFO SparkContext: Java version 17.0.12
26/04/22 11:11:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/22 11:11:20 INFO ResourceUtils: ==============================================================
26/04/22 11:11:20 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/22 11:11:20 INFO ResourceUtils: ==============================================================
26/04/22 11:11:20 INFO SparkContext: Submitted application: SparkRealRDD
26/04/22 11:11:20 INFO ResourceProfile: Default ResourceProfile created, executor resources: Map(cores -> name: cores, amount: 1, script: , vendor: , memory -> name: memory, amount: 1024, script: , vendor: , offHeap -> nam

ERROR StatusConsoleListener An exception occurred processing Appender console
 org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:165)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:134)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:125)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:89)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:683)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:641)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:624)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:560)
	at org.apache.logging.log4j.core.config.AwaitCompletionReliabil

import $ivy.$
import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.SparkSession@534ccb03
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@2babad7e
frases: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd1.sc:26

Ejercicio 8 — reduce para encontrar el máximo sin usar max()

Crea un RDD con los siguientes valores de ventas diarias (en euros) y usa reduce() para encontrar el valor más alto. No está permitido usar los métodos max() ni sum() de Spark: el cálculo debe hacerse con una función lambda dentro de reduce.

In [3]:
val ventasDiarias = sc.parallelize(List(1520.0, 890.5, 2340.0, 670.0, 1890.75, 3100.0, 450.25))
// Tu código aquí — usa reduce con una función lambda

val maximo = ventasDiarias.reduce((a, b) => if (a > b) a else b)
println(f"Venta diaria más alta: $$${maximo}%.2f")

Venta diaria más alta: $3100,00


ventasDiarias: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[6] at parallelize at cmd3.sc:1
maximo: Double = 3100.0

Ejercicio 9 — Contar elementos que cumplen una condición

A partir de un RDD con las edades de los visitantes de una web, calcula cuántos visitantes son mayores de edad (18 años o más) usando una combinación de filter y count(). Luego calcula también el porcentaje que representan sobre el total.

In [6]:
val edades = sc.parallelize(List(
  14, 23, 17, 35, 16, 28, 42, 15, 19, 31, 13, 25, 18, 22, 16, 45, 20, 12
))

// Tu código aquí
val total = edades.count()
println(s"Total de edades: $total")

val mayoresDeEdad = edades.filter(e => e >= 18)
val totalMayoresEdad = mayoresDeEdad.count()
println(s"Total de mayores de edad: $totalMayoresEdad")

val porcentajeMayoresEd = (totalMayoresEdad.toDouble / edades.count()) * 100
println(f"Porcentaje de mayores de edad: ${porcentajeMayoresEd}%.2f%%")


Total de edades: 18
Total de mayores de edad: 11
Porcentaje de mayores de edad: 61,11%


edades: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[11] at parallelize at cmd6.sc:1
total: Long = 18L
mayoresDeEdad: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[12] at filter at cmd6.sc:9
totalMayoresEdad: Long = 11L
porcentajeMayoresEd: Double = 61.111111111111114

Ejercicio 10 — WordCount de hashtags

Tienes el siguiente RDD con publicaciones de redes sociales. Usa flatMap, map y reduceByKey para contar cuántas veces aparece cada hashtag (palabras que empiezan por #). Muestra los resultados ordenados de mayor a menor frecuencia.

In [7]:
val publicaciones = sc.parallelize(List(
  "hoy aprendemos #spark con #scala en clase",
  "me encanta #scala es un lenguaje potente",
  "procesando big data con #spark y #hadoop",
  "#spark es más rápido que #hadoop en memoria",
  "aprendiendo #scala y #spark juntos hoy"
))
// Pista: después de flatMap, filtra solo las palabras que empiecen por "#"
// Tu código aquí
val palabraPublicacion = publicaciones.flatMap(p => p.split(" "))

val parPublicacion = palabraPublicacion.map(p => (p, 1))

val hashtag = parPublicacion.filter{ case (p, _) => p.startsWith("#") }

val conteoHashtag = hashtag.reduceByKey((a, b) => a + b)

val resultado = conteoHashtag.sortBy(_._2, ascending = false).collect()

println("Hashtags más populares:")
println("-" * 30)       
resultado.foreach{ case (tag, count) => println(s"$tag: $count") }

java.lang.IllegalArgumentException: Unable to create serializer "com.esotericsoftware.kryo.serializers.FieldSerializer" for class: java.lang.invoke.SerializedLambda

Ejercicio 11 — Verificar el linaje con toDebugString

Construye el siguiente pipeline de transformaciones y usa toDebugString para imprimir su linaje antes de ejecutar ninguna acción. Luego ejecuta collect() y comprueba que el resultado es correcto

In [8]:
val notas = sc.parallelize(List(4.5, 7.0, 3.2, 9.1, 5.5, 8.8, 2.0, 6.3, 7.7, 4.9))
// Paso 1: filtra las notas aprobadas (>= 5.0)
val notasAprobadas = notas.filter(n => n >= 5.0)

// Paso 2: multiplica cada nota por 10 para obtener la puntuación sobre 100
val notasPuntuacion = notasAprobadas.map(n => n * 10)

// Paso 3: filtra las que sean mayores o iguales a 70 (notable o superior)
val notasNotables = notasPuntuacion.filter(n => n >= 70)

// Tu código aquí — imprime toDebugString antes de collect()
println("Plan de ejecución RDD:")
println(notasNotables.toDebugString)
val resultado = notasNotables.collect()
println(s"Notas notables (>= 7.0): ${resultado.mkString(", ")}")


Plan de ejecución RDD:
(8) MapPartitionsRDD[24] at filter at cmd8.sc:9 []
 |  MapPartitionsRDD[23] at map at cmd8.sc:6 []
 |  MapPartitionsRDD[22] at filter at cmd8.sc:3 []
 |  ParallelCollectionRDD[21] at parallelize at cmd8.sc:1 []
ERROR StatusConsoleListener An exception occurred processing Appender console
 org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:165)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:134)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:125)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:89)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:683)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.ja

notas: org.apache.spark.rdd.RDD[Double] = ParallelCollectionRDD[21] at parallelize at cmd8.sc:1
notasAprobadas: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[22] at filter at cmd8.sc:3
notasPuntuacion: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[23] at map at cmd8.sc:6
notasNotables: org.apache.spark.rdd.RDD[Double] = MapPartitionsRDD[24] at filter at cmd8.sc:9
resultado: Array[Double] = Array(70.0, 91.0, 88.0, 77.0)

Ejercicio 12 — take vs collect

Crea un RDD con los números del 1 al 1000 y aplica una transformación que calcule el cubo de cada número (n * n * n). Luego usa take(5) para mostrar solo los cinco primeros resultados sin traer los 1000 elementos al Driver. Como segundo paso, usa count() para confirmar cuántos elementos tiene el RDD resultante.

In [9]:
val rango = sc.parallelize(1 to 1000)
// Tu código aquí
val rangoAlCubo = rango.map(n => n * n * n)

val resultado = rangoAlCubo.take(5)
println(s"Los primeros 5 números al cubo: ${resultado.mkString(", ")}")

val contar = rangoAlCubo.count()
println(s"Total de números al cubo: $contar")

Los primeros 5 números al cubo: 1, 8, 27, 64, 125
Total de números al cubo: 1000


rango: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[25] at parallelize at cmd9.sc:1
rangoAlCubo: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[26] at map at cmd9.sc:3
resultado: Array[Int] = Array(1, 8, 27, 64, 125)
contar: Long = 1000L

Ejercicio 13 — Cachear un RDD reutilizado

Crea un RDD con los números del 1 al 500.000. Aplica un filter que se quede solo con los múltiplos de 3. Persiste ese RDD en memoria antes de ejecutar ninguna acción. Luego realiza dos acciones consecutivas sobre él: count() y sum(). Finalmente, libera el caché con unpersist().

El objetivo es observar en el Spark UI que la segunda acción reutiliza el RDD cacheado en lugar de recalcularlo.

In [10]:
val granRdd = sc.parallelize(1 to 500000)
// Tu código aquí:
// 1. Crea el RDD de múltiplos de 3
val multiplosDe3 = granRdd.filter(n => n % 3 == 0)
// 2. Persiste el RDD
multiplosDe3.persist()
// 3. count() — primera acción
val cantidad = multiplosDe3.count()
println(s"Cantidad de múltiplos de 3: $cantidad")
// 4. sum()   — segunda acción (debe usar el caché)
val suma = multiplosDe3.sum()
println(s"Suma de múltiplos de 3: $suma")
// 5. unpersist()
multiplosDe3.unpersist()

Cantidad de múltiplos de 3: 166666
Suma de múltiplos de 3: 4.1666583333E10


granRdd: org.apache.spark.rdd.RDD[Int] = ParallelCollectionRDD[27] at parallelize at cmd10.sc:1
multiplosDe3: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[28] at filter at cmd10.sc:4
res10_2: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[28] at filter at cmd10.sc:4
cantidad: Long = 166666L
suma: Double = 4.1666583333E10
res10_7: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[28] at filter at cmd10.sc:4

In [11]:
// Intenta crear una SEGUNDA SparkSession con otro nombre
val spark2 = SparkSession.builder()
  .appName("SesionNueva-Que-No-Existira")
  .master("local[*]")
  .getOrCreate()

// ¿Cuál es el nombre real de spark2?
println(s"Nombre de spark2: ${spark2.sparkContext.appName}")
println(s"¿spark y spark2 son la misma sesión?: ${spark eq spark2}")

ERROR StatusConsoleListener An exception occurred processing Appender console
 org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:165)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:134)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:125)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:89)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:683)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:641)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:624)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerConfig.java:560)
	at org.apache.logging.log4j.core.config.AwaitCompletionReliabil

spark2: SparkSession = org.apache.spark.sql.SparkSession@534ccb03

Ejercicio 15 — Pipeline completo: de datos en bruto a resumen

Este ejercicio integra todo lo visto en la sesión. Tienes el siguiente RDD con registros de accesos a un servidor web (formato: "IP MÉTODO RUTA CÓDIGO_HTTP"). Construye un pipeline que:

Filtre solo los registros con código HTTP 404 (página no encontrada).
Extraiga únicamente la ruta de cada registro fallido.
Cuente cuántas veces aparece cada ruta con reduceByKey.
Ordene las rutas por número de accesos fallidos de mayor a menor.
Muestre el resultado con take(5).

In [13]:
val logs = sc.parallelize(List(
  "192.168.1.1 GET /inicio 200",
  "192.168.1.2 GET /productos 404",
  "192.168.1.3 POST /login 200",
  "192.168.1.4 GET /productos 404",
  "192.168.1.5 GET /contacto 404",
  "192.168.1.6 GET /inicio 200",
  "192.168.1.7 GET /productos 404",
  "192.168.1.8 GET /blog 404",
  "192.168.1.9 GET /contacto 404",
  "192.168.1.10 GET /productos 404"
))

// Pista: cada línea se puede dividir con .split(" ")
// El código HTTP es el último elemento del array resultante (índice 3)
// La ruta es el tercer elemento (índice 2)
// Tu código aquí

val filteredLogs = logs.filter(line => line.split(" ")(3) == "404")
val rutas404 = filteredLogs.map(line => line.split(" ")(2))
val conteoRutas404 = rutas404.map(ruta => (ruta, 1)).reduceByKey((a, b) => a + b)
val resultado = conteoRutas404.sortBy(_._2, ascending = false).collect()
println("Rutas con más errores 404:")
resultado.foreach{ case (ruta, count) => println(s"$ruta: $count accesos fallidos") }

java.lang.IllegalArgumentException: Unable to create serializer "com.esotericsoftware.kryo.serializers.FieldSerializer" for class: java.lang.invoke.SerializedLambda

🏢 Caso de Empresa

In [20]:
val reproducciones = List(
  ("U001", "La Casa de Papel",         "Serie",        47),
  ("U002", "Dune: Parte Dos",          "Película",     95),
  ("U003", "Cosmos: Mundos Posibles",  "Documental",   42),
  ("U001", "Dune: Parte Dos",          "Película",    107),
  ("U004", "La Casa de Papel",         "Serie",        52),
  ("U005", "El Problema de los 3 Cuerpos", "Serie",   58),
  ("U002", "Cosmos: Mundos Posibles",  "Documental",   38),
  ("U006", "La Casa de Papel",         "Serie",        49),
  ("U003", "El Problema de los 3 Cuerpos", "Serie",   61),
  ("U007", "Dune: Parte Dos",          "Película",     88),
  ("U005", "La Casa de Papel",         "Serie",        44),
  ("U008", "Cosmos: Mundos Posibles",  "Documental",   55),
  ("U004", "El Problema de los 3 Cuerpos", "Serie",   63),
  ("U009", "Dune: Parte Dos",          "Película",    112),
  ("U006", "Cosmos: Mundos Posibles",  "Documental",   29),
  ("U010", "La Casa de Papel",         "Serie",        51),
  ("U007", "El Problema de los 3 Cuerpos", "Serie",   57),
  ("U008", "La Casa de Papel",         "Serie",        46),
  ("U001", "Cosmos: Mundos Posibles",  "Documental",   33),
  ("U009", "La Casa de Papel",         "Serie",        48),
  ("U010", "Dune: Parte Dos",          "Película",    102),
  ("U002", "El Problema de los 3 Cuerpos", "Serie",   60),
  ("U003", "Dune: Parte Dos",          "Película",     91),
  ("U011", "Cosmos: Mundos Posibles",  "Documental",   47),
  ("U012", "La Casa de Papel",         "Serie",        53)
)

val reproduccionesRDD = sc.parallelize(reproducciones)

println(s"Sesión de Spark configurada : ${sc.appName}")
println(s"Tipo del RDD:       ${reproduccionesRDD.getClass.getSimpleName}")
println(s"Número de partes:   ${reproduccionesRDD.getNumPartitions}")
println(s"¿Se ha ejecutado?:  No — solo hemos definido el plan")

//Tarea 2 — ¿Cuántas reproducciones completas hubo ayer?
val reproducionMayor45 = reproduccionesRDD.filter{ case (_, _, _, minutos_vistos) => minutos_vistos > 45 }
val totalReproduccionesCompletas = reproducionMayor45.count()
println("Informe de reproducciones completas:")
println("-" * 40)
println(s"Total de reproducciones registradas: ${reproduccionesRDD.count()}")
println(s"Total de reproducciones completas (> 45 min): $totalReproduccionesCompletas")
val porcentaje = (totalReproduccionesCompletas.toDouble / reproduccionesRDD.count()) * 100
println(f"Porcentaje de reproducciones completas: ${porcentaje}%.2f%%")



Sesión de Spark configurada : SparkRealRDD
Tipo del RDD:       ParallelCollectionRDD
Número de partes:   8
¿Se ha ejecutado?:  No — solo hemos definido el plan
ERROR StatusConsoleListener An exception occurred processing Appender console
 org.apache.logging.log4j.core.appender.AppenderLoggingException: java.lang.AssertionError: assertion failed
	at org.apache.logging.log4j.core.config.AppenderControl.tryCallAppender(AppenderControl.java:165)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender0(AppenderControl.java:134)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppenderPreventRecursion(AppenderControl.java:125)
	at org.apache.logging.log4j.core.config.AppenderControl.callAppender(AppenderControl.java:89)
	at org.apache.logging.log4j.core.config.LoggerConfig.callAppenders(LoggerConfig.java:683)
	at org.apache.logging.log4j.core.config.LoggerConfig.processLogEvent(LoggerConfig.java:641)
	at org.apache.logging.log4j.core.config.LoggerConfig.log(LoggerCo

reproducciones: List[(String, String, String, Int)] = List(
  ("U001", "La Casa de Papel", "Serie", 47),
  ("U002", "Dune: Parte Dos", "Película", 95),
  ("U003", "Cosmos: Mundos Posibles", "Documental", 42),
  ("U001", "Dune: Parte Dos", "Película", 107),
  ("U004", "La Casa de Papel", "Serie", 52),
  ("U005", "El Problema de los 3 Cuerpos", "Serie", 58),
  ("U002", "Cosmos: Mundos Posibles", "Documental", 38),
  ("U006", "La Casa de Papel", "Serie", 49),
  ("U003", "El Problema de los 3 Cuerpos", "Serie", 61),
  ("U007", "Dune: Parte Dos", "Película", 88),
  ("U005", "La Casa de Papel", "Serie", 44),
  ("U008", "Cosmos: Mundos Posibles", "Documental", 55),
  ("U004", "El Problema de los 3 Cuerpos", "Serie", 63),
  ("U009", "Dune: Parte Dos", "Película", 112),
  ("U006", "Cosmos: Mundos Posibles", "Documental", 29),
  ("U010", "La Casa de Papel", "Serie", 51),
  ("U007", "El Problema de los 3 Cuerpos", "Serie", 57),
  ("U008", "La Casa de Papel", "Serie", 46),
  ("U001", "Cosmos: Mund

In [ ]:
//Tarea 3 — ¿Qué género consumió más minutos en total?

val generoMinutos = reproduccionesRDD.map{ case (_, _, genero, minutos_vistos) => (genero, minutos_vistos) }
val minutosPorGenero = generoMinutos.reduceByKey((a, b) => a + b)
val resultado = minutosPorGenero.sortBy(_._2, ascending = false).collect()
println("Géneros ordenados por minutos consumidos:")
println("-" * 40)
resultado.foreach{ case (genero, minutos) => println(s"$genero: $minutos minutos") }

In [ ]:
//Tarea 4 — ¿Cuál es el contenido más visto y cuántos usuarios únicos lo vieron?
val contenidoUsuarios = reproduccionesRDD.map{ case (_, titulo, _, _) => (titulo, 1) }
val minutosPorContenido = contenidoUsuarios.reduceByKey((a, b) => a + b)
val resultado = minutosPorContenido.reduce((a, b) => if (a._2 > b._2) a else b)
println("Contenido más visto:")
println("-" * 40)
println(s"Título: ${resultado._1}")
println(s"Minutos vistos: ${resultado._2}")

//val filteredLogs = logs.filter(line => line.split(" ")(3) == "404")
val contenidoUsuariosUnicos = reproduccionesRDD.filter (r => r._2 == resultado._1)
                                               .map(r => r._1)
                                               .distinct()
                                               .count()
println(s"Usuarios únicos que vieron el contenido: $contenidoUsuariosUnicos")


Carmen tiene una petición extra: identifica qué usuario individual acumuló más minutos totales de visualización en el día. Necesitarás transformar el RDD en pares (usuario_id, minutos), sumar los minutos por usuario con reduceByKey, y luego usar reduce para encontrar el par con el valor más alto.

In [ ]:
val usuarioUnicoTotal = reproduccionesRDD.map{ case (usuario_id, _, _, minutos_vistos) => (usuario_id, minutos_vistos) }
val minutosPorUsuario = usuarioUnicoTotal.reduceByKey((a, b) => a + b)
val resultado = minutosPorUsuario.reduce((a, b) => if (a._2 > b._2) a else b)
println("Usuario que más minutos vio:")
println("-" * 40)  
println(s"ID de usuario: ${resultado._1}")
println(s"Minutos vistos: ${resultado._2}")
